In [1]:
import os

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

In [2]:
from pathlib import Path

import nibabel as nib

image_path = Path("/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_bold.nii.gz")

In [3]:
import json

import numpy as np

sidecar = json.loads(
    (image_path.parent / image_path.name.removesuffix(".gz").removesuffix(".nii"))
    .with_suffix(".json")
    .read_text()
)
slice_timing = np.asanyarray(sidecar["SliceTiming"])

In [4]:
import scipy.stats

slice_ranks = scipy.stats.rankdata(
    (slice_timing * 10000).astype(int), method="dense"
).astype(int)

In [5]:
frames_per_second = np.reciprocal(
    sidecar["RepetitionTime"] / np.unique(slice_ranks).size
)
frames_per_second

np.float64(11.25)

In [6]:
volume_count = round(60 / sidecar["RepetitionTime"])
volume_count

75

In [7]:
slice_timing_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5_task-restPA_run-01_bold_valid_tshift.nii.gz")
image = nib.nifti1.load(slice_timing_path) # pyright: ignore[reportAttributeAccessIssue]

In [8]:
data = image.get_fdata()
data.shape

(104, 104, 72, 488)

In [9]:
import nitransforms
from fmriprep.interfaces.resampling import load_transforms, reconstruct_fieldmap
from sdcflows.transform import B0FieldTransform
from sdcflows.utils.tools import ensure_positive_cosines

derivatives_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5")

motion_correction_transforms = nitransforms.linear.load(
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-orig_to-boldref_mode-image_desc-hmc_xfm.txt"
)
if not isinstance(motion_correction_transforms, nitransforms.linear.LinearTransformsMapping):
    raise TypeError

coeffs = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "fmap/sub-NDARINVXP963GZ5_fmapid-SpinEchoFieldMapfmap_desc-coeff_fieldmap.nii.gz"
)
boldref = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_desc-hmc_boldref.nii.gz"
)
fmap_reference = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "fmap/sub-NDARINVXP963GZ5_fmapid-SpinEchoFieldMapfmap_desc-epi_fieldmap.nii.gz"
)
target, _ = ensure_positive_cosines(boldref)

b0_field_transform = B0FieldTransform(coeffs=coeffs)  # pyright: ignore[reportCallIssue]
b0_field_transform.mapped = reconstruct_fieldmap(
    coefficients=[coeffs],
    fmap_reference=fmap_reference,
    target=target,
    transforms=load_transforms(
        [
            derivatives_path
            / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-boldref_to-SpinEchoFieldMapfmap_mode-image_desc-fmap_xfm.txt"
        ],
        [True],
    ),
)

pe_dir = sidecar["PhaseEncodingDirection"]
flipped_pe = pe_dir[0] if pe_dir.endswith("-") else pe_dir + "-"
mapped = b0_field_transform.mapped
forward_displacement_image = b0_field_transform.to_displacements(
    ro_time=sidecar["TotalReadoutTime"],
    pe_dir=pe_dir,
    itk_format=True,
)
forward_displacement_image.to_filename("forward_displacement.nii.gz")

In [10]:
import ants

forward_field = ants.image_read("forward_displacement.nii.gz")
inverse_field_initial_estimate = forward_field.new_image_like(  # pyright: ignore[reportAttributeAccessIssue]
    forward_field.numpy() * -1.0
)
backward_field = ants.invert_displacement_field(
    forward_field, inverse_field_initial_estimate, maximum_number_of_iterations=1_000
)
backward_path = Path("backward_displacement.nii.gz")
ants.image_write(backward_field, backward_path.as_posix())

In [11]:
import ants
import pandas as pd


def warp_points(
    motion_correction_transform: nitransforms.linear.Affine, points: np.ndarray
) -> np.ndarray:
    motion_correction_transform_path = Path("motion_correction_transform.mat")
    motion_correction_transform.to_filename(motion_correction_transform_path, fmt="itk")

    points = nib.affines.apply_affine(image.affine, points)  # pyright: ignore[reportAttributeAccessIssue]

    points_data_frame = pd.DataFrame(points, columns=["x", "y", "z"])
    points_data_frame["x"] *= -1.0
    points_data_frame["y"] *= -1.0
    points_data_frame = ants.apply_transforms_to_points(
        dim=3,
        points=points_data_frame,
        transformlist=[
            motion_correction_transform_path.as_posix(),
            backward_path.as_posix(),
        ],
        whichtoinvert=[
            True,
            False,
        ],
    )
    points_data_frame["x"] *= -1.0
    points_data_frame["y"] *= -1.0

    return nib.affines.apply_affine(  # pyright: ignore[reportAttributeAccessIssue]
        np.linalg.inv(image.affine), points_data_frame.to_numpy()
    )

In [ ]:
from collections.abc import MutableSequence
from typing import Any

import imageio.v3
import pyvista as pv
from tqdm.auto import tqdm

vox2ras = image.affine
ras2vox = np.linalg.inv(vox2ras)

plot_slices = (
    ("x", "yz"),
    ("y", "xz"),
    ("z", "xy"),
)
plot_voxel = np.array([52, 48, 21])

window_size = [3840, 2160]
aspect = (window_size[0] / len(plot_slices)) / window_size[1]

fixed_cameras: MutableSequence[Any] = [None, None, None]
volume_center = None
scales: MutableSequence[float | None] = [None, None, None]

writer = imageio.get_writer(
    "susceptibility-distortion-correction.webm",
    format="FFMPEG",  # pyright: ignore[reportArgumentType]
    codec="vp9",
    fps=frames_per_second,
    output_params=["-crf", "32"],
)

grid = pv.ImageData(
    dimensions=np.array(data.shape[:3]) + 1,
    spacing=(1.0, 1.0, 1.0),
    origin=(-0.5, -0.5, -0.5),
)

current_data = np.zeros_like(data[..., 0])
cumulative_data = np.zeros_like(data[..., 0])

last_rank = slice_ranks.max()

volume_center = plot_voxel

for volume_index in tqdm(range(volume_count)):  # data.shape[-1]
    for rank in np.sort(np.unique(slice_ranks))[-2:]:
        is_last_slice = last_rank == rank
        if is_last_slice:
            current_data[:] = 0.25
        else:
            current_data[:] = 0.0

        cumulative_data[..., :] = data[..., :, volume_index]

        grid.cell_data["current"] = current_data.flatten(order="F")
        grid.cell_data["cumulative"] = cumulative_data.flatten(order="F")

        motion_correction_transform = motion_correction_transforms[volume_index] # pyright: ignore[reportIndexIssue]
        transformed_grid = grid.cast_to_structured_grid()
        transformed_grid.points = warp_points(motion_correction_transform, grid.points)

        transformed_grid = transformed_grid.triangulate()

        plotter = pv.Plotter(
            shape=(1, len(plot_slices)),
            off_screen=True,
            window_size=window_size,
            border=False,
        )
        for i, (normal, camera_position) in enumerate(plot_slices):
            plotter.subplot(0, i)

            plotter.set_background("black")  # pyright: ignore[reportArgumentType]

            intersection = transformed_grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
            plotter.add_mesh(
                intersection,
                scalars=scalars,
                cmap="gray",
                lighting=False,
                show_edges=False,
                show_scalar_bar=False,
            )
            plotter.add_mesh(
                intersection,
                scalars="current",
                opacity="current",  # pyright: ignore[reportArgumentType]
                cmap="Blues",
                clim=(0.0, 0.25),
                lighting=False,
                show_edges=False,
                show_scalar_bar=False,
            )

            edges = intersection.threshold(
                [10.0, np.inf], scalars="cumulative"
            ).extract_all_edges()
            if edges.n_faces:
                plotter.add_mesh(edges, color="white", line_width=0.5)  # pyright: ignore[reportArgumentType]
            plotter.enable_anti_aliasing("ssaa")

            if scales[i] is None:
                bounds = intersection.bounds
                sizes = [
                    bounds.x_max - bounds.x_min,
                    bounds.y_max - bounds.y_min,
                    bounds.z_max - bounds.z_min,
                ]
                sizes.pop(i)
                scale = (
                    max(sizes[1], sizes[0] / aspect)
                    * 0.5
                    * {"x": 0.9, "y": 0.8, "z": 0.7}[normal]
                )
                scales[i] = scale
            if fixed_cameras[i] is None:
                plotter.camera_position = camera_position  # pyright: ignore[reportAttributeAccessIssue]
                fixed_cameras[i] = [tuple(v) for v in plotter.camera_position]
            plotter.camera_position = fixed_cameras[i]

            plotter.enable_parallel_projection()  # pyright: ignore[reportCallIssue]
            plotter.camera.parallel_scale = scales[i]
            plotter.reset_camera_clipping_range()

        plotter.screenshot("screenshot.png")
        plotter.close()

        frame = imageio.v3.imread("screenshot.png")

        if is_last_slice:
            writer.append_data(frame)
        else:
            for _ in range(np.sort(np.unique(slice_ranks))[-2]):
                writer.append_data(frame)

writer.close()